# Tech Challenge – FIAP (Data Analytics) – Fase 4
## Predição de Nível de Obesidade

Este notebook executa:
- Carregamento e entendimento dos dados
- Feature engineering (BMI) + arredondamento de variáveis ordinais
- Pipeline de pré-processamento (OneHotEncoder) + modelo (Random Forest)
- Avaliação (acurácia, matriz de confusão, relatório)
- Exportação de artefatos para Power BI


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('obesity_dataset.csv.csv')
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape, df.dtypes

## Feature engineering
- BMI = Weight / Height²
- Variáveis ordinais com decimais são arredondadas: FCVC, NCP, CH2O, FAF, TUE

In [ ]:
df2 = df.copy()
df2['BMI'] = df2['Weight']/(df2['Height']**2)
ord_cols = ['FCVC','NCP','CH2O','FAF','TUE']
for c in ord_cols:
    if c in df2.columns:
        df2[c] = df2[c].round().astype(int)

df2[ord_cols + ['BMI']].describe()

## Treinamento do modelo (Pipeline)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

TARGET = 'Obesity'
X = df2.drop(columns=[TARGET])
y = df2[TARGET]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample'
)

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

## Avaliação

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df

In [ ]:
print(classification_report(y_test, pred))

## Exportação de artefatos
Gera arquivos em `/powerbi_assets` para você importar no Power BI.

In [ ]:
import joblib

ASSETS = Path('powerbi_assets')
ASSETS.mkdir(exist_ok=True)

# Confusion Matrix
cm_df.to_csv(ASSETS/'confusion_matrix.csv', index=True)

# Classification report
rep = classification_report(y_test, pred, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(rep).T
rep_df.to_csv(ASSETS/'classification_report.csv', index=True)

# Class distribution
(dist := y.value_counts().rename_axis('class').reset_index(name='count'))
dist['percent'] = dist['count']/dist['count'].sum()
dist.to_csv(ASSETS/'class_distribution.csv', index=False)

# Feature importance (top 30)
ohe = clf.named_steps['preprocess'].named_transformers_['cat']
feature_names = list(ohe.get_feature_names_out(cat_cols)) + num_cols
importances = clf.named_steps['model'].feature_importances_
fi = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
fi.head(30).to_csv(ASSETS/'feature_importance_top30.csv', index=False)

# Save model
joblib.dump(clf, 'model.joblib')

print('OK! Arquivos gerados em:', ASSETS.resolve())
print('Modelo salvo em: model.joblib')